# Pins Face Recognition — Paper Tables

Generates the three LaTeX tables for Pins Face Recognition that appear in the paper:

1. **`pins_results_table.tex`** (main): $\Delta\mathrm{Acc}_{D_{\mathrm{f}}'}$, $\Delta\mathrm{Acc}_{D_{\mathrm{r}}'}$, $\mathrm{Layer}$
2. **`pins_appendix_table.tex`** (appendix): $\mathrm{Activ}_{D_{\mathrm{f}}'}$, $\mathrm{Activ}_{D_{\mathrm{r}}'}$, $\Delta\mathrm{MIA}$
3. **`pins_raw_results_table.tex`** (appendix raw values): raw $\mathrm{Acc}$ and $\mathrm{MIA}$ for $M_{\mathrm{u}}$ and $M_{\mathrm{r}}$

Loads only PinsFaceRecognition CSVs (ResNet18 + ViT, full-class + random) and overlays the Y56 retrain values from W&B for ResNet18 PinsFace.


In [ ]:
import pandas as pd
import zipfile
from pathlib import Path

# Locate the CSV folder
folder_candidates = [
    Path("wandb_metrics_summary"),
    Path("src/utils/wandb_utils/results_extraction/wandb_metrics_summary"),
    Path("../results_extraction/wandb_metrics_summary"),
]
csv_base = next((p for p in folder_candidates if p.is_dir()), None)

if csv_base is None:
    zip_candidates = [Path("wandb_metrics_summary.zip")]
    zip_path = next((p for p in zip_candidates if p.is_file()), None)
    if zip_path is None:
        raise FileNotFoundError(
            "Could not find wandb_metrics_summary/ or wandb_metrics_summary.zip. "
            "Run the W&B export pipeline or extract the zip next to this notebook."
        )
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(zip_path.parent)
    csv_base = zip_path.parent / "wandb_metrics_summary"

print(f"Using data from: {csv_base.resolve()}")

# Load PinsFaceRecognition CSVs only
pins_dir = csv_base / "PinsFaceRecognition"
assert pins_dir.is_dir(), f"PinsFaceRecognition folder not found at {pins_dir}"

dataframes = {}
for csv_file in sorted(pins_dir.glob("*.csv")):
    dataframes[csv_file.stem] = pd.read_csv(csv_file)
    print(f"  Loaded {csv_file.stem}: {len(dataframes[csv_file.stem])} rows")
print(f"\nTotal Pins projects loaded: {len(dataframes)}")

In [ ]:
# === Overlay Y56 retrain results for ResNet18 PinsFaceRecognition ===
#
# Pulls retrain runs from the Y56_UNLEARNING_* W&B projects and replaces the
# corresponding retrain rows in the loaded dataframes.

import os
import re
import warnings
import wandb

# Parse .env from common notebook cwd locations
for _envp in [".env", "../.env", "../../.env", "../../../.env", "../../../../.env"]:
    if os.path.isfile(_envp):
        for _line in open(_envp):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
        break

WANDB_PREFIX = "Y56_UNLEARNING"
WANDB_SUFFIX = "_precision_32-true_no_dist"

patch_targets = [
    ("ResNet18", "PinsFaceRecognition", "fullclass", "1"),
    ("ResNet18", "PinsFaceRecognition", "fullclass", "10"),
    ("ResNet18", "PinsFaceRecognition", "fullclass", "20"),
    ("ResNet18", "PinsFaceRecognition", "fullclass", "30"),
    ("ResNet18", "PinsFaceRecognition", "fullclass", "40"),
    ("ResNet18", "PinsFaceRecognition", "random_", "0.001"),
]


def _project_name(model, dataset, strategy, target):
    if strategy == "random_":
        return f"{WANDB_PREFIX}_{model}_{dataset}_random__{target}perc{WANDB_SUFFIX}"
    return f"{WANDB_PREFIX}_{model}_{dataset}_{strategy}_{target}{WANDB_SUFFIX}"


def _df_key(model, dataset, strategy, target):
    if strategy == "random_":
        return f"{model}_{dataset}_random__{target}perc"
    return f"{model}_{dataset}_{strategy}_{target}"


def _flatten_summary(d, parent=""):
    out = {}
    for k, v in d.items():
        key = f"{parent}.{k}" if parent else k
        if isinstance(v, dict):
            if "final_value" in v and not isinstance(v["final_value"], dict):
                out[key] = v["final_value"]
            else:
                out.update(_flatten_summary(v, key))
        else:
            out[key] = v
    return out


api = wandb.Api()
_entity = os.getenv("WANDB_ENTITY") or api.default_entity
# Match three run-name formats in order of specificity:
#   1. retrain_tseed{T}_useed{U}_eseed{E}   (K>1 eval; capture the useed = s_u)
#   2. retrain_tseed{T}_useed{U}            (K=1; capture useed)
#   3. retrain_seed{U}                      (oldest fallback; capture seed)
# Patched rows store the *unlearning* seed (s_u) in the dataframe's 'seed' column
# so K>1 eval runs collapse to the same row as their K=1 counterpart by design.
_SEED_RE = re.compile(r"^retrain_(?:tseed\d+_useed(\d+)(?:_eseed\d+)?|seed(\d+))$")

_total_replaced = 0
_total_added = 0
print(f"Pulling Y56 retrain runs (entity={_entity}) ...")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    for _model, _ds, _strat, _tgt in patch_targets:
        _proj = _project_name(_model, _ds, _strat, _tgt)
        _key = _df_key(_model, _ds, _strat, _tgt)
        if _key not in dataframes:
            print(f"  WARN: no existing df for {_key} - skipping")
            continue
        _df = dataframes[_key]
        _meta = {
            "project",
            "model",
            "dataset",
            "strategy",
            "target",
            "method",
            "seed",
            "run_id",
            "state",
        }
        _metric_cols = [c for c in _df.columns if c not in _meta]
        try:
            _runs = list(api.runs(f"{_entity}/{_proj}"))
        except Exception as e:
            print(f"  ERROR {_proj}: {type(e).__name__}: {e}")
            continue
        _retrain_runs = [r for r in _runs if r.name.startswith("retrain_")]
        if not _retrain_runs:
            print(f"  WARN: {_proj} has no retrain runs")
            continue
        _new_rows = []
        for _r in _retrain_runs:
            _m = _SEED_RE.match(_r.name)
            if not _m:
                continue
            # Either group 1 (tseed/useed[/eseed] form) or group 2 (old seed form) matches.
            _seed = int(_m.group(1) or _m.group(2))
            _flat = _flatten_summary(_r.summary._json_dict or {})
            if _strat == "random_":
                _tgt_value = f"{_tgt}perc"
            else:
                _tgt_value = int(_tgt) if _tgt.isdigit() else _tgt
            _row = {
                "project": _proj,
                "model": _model,
                "dataset": _ds,
                "strategy": _strat,
                "target": _tgt_value,
                "method": "retrain",
                "seed": _seed,
                "run_id": _r.id,
                "state": _r.state,
            }
            for _wk, _v in _flat.items():
                _col = _wk.replace(".", "_")
                if _col in _metric_cols:
                    _row[_col] = _v
            for _c in _metric_cols:
                _row.setdefault(_c, None)
            _new_rows.append(_row)
        if not _new_rows:
            print(f"  WARN: {_proj} - no parseable retrain rows")
            continue
        _new_df = pd.DataFrame(_new_rows)
        _n_old = int((_df["method"] == "retrain").sum())
        _df_other = _df[_df["method"] != "retrain"].copy()
        dataframes[_key] = pd.concat([_df_other, _new_df], ignore_index=True)
        print(
            f"  OK  {_key}: replaced {_n_old} retrain rows -> {len(_new_df)} fresh Y56 rows"
        )
        _total_replaced += _n_old
        _total_added += len(_new_df)

print(
    f"\nPatched: {_total_replaced} old retrain rows -> {_total_added} Y56 retrain rows."
)

In [ ]:
# Unified raw dataframe (used by the raw-values table)
df_all = pd.concat(dataframes.values(), ignore_index=True)
print(f"df_all (raw): {len(df_all)} rows, {len(df_all.columns)} columns")

# Rename metric columns to the short forms
COLUMN_RENAME = {
    "accuracy_test_unlearning_method_whole_acc": "acc",
    "accuracy_test_unlearning_method_retain_acc": "retain_acc",
    "accuracy_test_unlearning_method_forget_acc": "forget_acc",
    "loss_test_unlearning_method_whole_loss": "loss",
    "loss_test_unlearning_method_retain_loss": "retain_loss",
    "loss_test_unlearning_method_forget_loss": "forget_loss",
    "membership_inference_attack_test_unlearning_method_mia": "mia",
    "activation_distance_test_unlearning_method_activation_distance_whole": "activation_distance_whole",
    "activation_distance_test_unlearning_method_activation_distance_retain": "activation_distance_retain",
    "activation_distance_test_unlearning_method_activation_distance_forget": "activation_distance_forget",
    "layerwise_distance_unlearning_method_layerwise_distance": "layerwise_distance",
}
df_all.rename(columns=COLUMN_RENAME, inplace=True)

meta_cols = [
    "project",
    "model",
    "dataset",
    "strategy",
    "target",
    "method",
    "seed",
    "run_id",
    "state",
]
metric_cols = [c for c in df_all.columns if c not in meta_cols]

# Per-seed normalization: subtract retrain values for delta-style metrics
SUBTRACT_METRICS = [
    "forget_acc",
    "retain_acc",
    "acc",
    "loss",
    "retain_loss",
    "forget_loss",
    "mia",
]
retrain_df = df_all[df_all["method"] == "retrain"].copy()
retrain_lookup = retrain_df.set_index(
    ["model", "dataset", "strategy", "target", "seed"]
)[metric_cols]

df_normalized = df_all[df_all["method"] != "retrain"].copy()
for metric in SUBTRACT_METRICS:
    if metric in df_normalized.columns:
        retrain_vals = retrain_lookup[[metric]].rename(
            columns={metric: f"{metric}_retrain"}
        )
        df_normalized = df_normalized.merge(
            retrain_vals.reset_index(),
            on=["model", "dataset", "strategy", "target", "seed"],
            how="left",
        )
        df_normalized[metric] = (
            df_normalized[metric] - df_normalized[f"{metric}_retrain"]
        )
        df_normalized.drop(columns=[f"{metric}_retrain"], inplace=True)

# Full-class stats: average over 5 forget classes per seed, then mean/std across 10 seeds
df_avg_targets = (
    df_normalized.groupby(["model", "dataset", "strategy", "method", "seed"])[
        metric_cols
    ]
    .mean()
    .reset_index()
)
df_stats = (
    df_avg_targets.groupby(["model", "dataset", "strategy", "method"])[metric_cols]
    .agg(["mean", "std"])
    .reset_index()
)
df_stats.columns = [f"{c[0]}_{c[1]}" if c[1] else c[0] for c in df_stats.columns]

# Per-target stats for random (used to filter to the 0.1% forget percentage)
_sub_random = df_normalized[df_normalized["strategy"] == "random_"]
df_random_by_target = (
    _sub_random.groupby(["model", "dataset", "method", "target"])[metric_cols]
    .agg(["mean", "std"])
    .reset_index()
)
df_random_by_target.columns = [
    f"{c[0]}_{c[1]}" if c[1] else c[0] for c in df_random_by_target.columns
]

print(f"df_stats:             {len(df_stats)} rows")
print(f"df_random_by_target:  {len(df_random_by_target)} rows")

In [ ]:
# === Constants and helper functions for the Pins tables ===

PINS_METHOD_ORDER_FC = [
    "finetune",
    "bad_teacher",
    "unsir",
    "random_labeling",
    "ssd",
    "lfssd",
]
PINS_METHOD_DISPLAY = {
    "finetune": "FT",
    "bad_teacher": "BadT",
    "unsir": "UNSIR",
    "random_labeling": "RL",
    "ssd": "SSD",
    "lfssd": "LFSSD",
}
PINS_DISPLAY_DECIMALS = 2

MAIN_METRICS = [
    ("forget_acc", "abs_min"),
    ("retain_acc", "abs_min"),
    ("layerwise_distance", "min"),
]
APPENDIX_METRICS = [
    ("activation_distance_forget", "min"),
    ("activation_distance_retain", "min"),
    ("mia", "abs_min"),
]
RAW_METRICS = ["forget_acc", "retain_acc", "mia"]


def _pins_lookup(model, strategy, method, metric):
    # (mean, std) for a (model, strategy, method, metric). Random uses 0.1% forget only.
    if strategy == "random_":
        sub = df_random_by_target[
            (df_random_by_target["dataset"] == "PinsFaceRecognition")
            & (df_random_by_target["model"] == model)
            & (df_random_by_target["method"] == method)
            & (df_random_by_target["target"] == "0.001perc")
        ]
        if sub.empty:
            return float("nan"), float("nan")
        return float(sub[f"{metric}_mean"].iloc[0]), float(sub[f"{metric}_std"].iloc[0])
    sub = df_stats[
        (df_stats["dataset"] == "PinsFaceRecognition")
        & (df_stats["model"] == model)
        & (df_stats["strategy"] == strategy)
        & (df_stats["method"] == method)
    ]
    if sub.empty:
        return float("nan"), float("nan")
    return float(sub[f"{metric}_mean"].iloc[0]), float(sub[f"{metric}_std"].iloc[0])


def _pins_methods_avail(model, strategy):
    # Restrict to the 6 paper-table methods; exclude UNSIR from random.
    methods = [
        m for m in PINS_METHOD_ORDER_FC if not (strategy == "random_" and m == "unsir")
    ]
    return [
        m
        for m in methods
        if not df_stats[
            (df_stats["dataset"] == "PinsFaceRecognition")
            & (df_stats["model"] == model)
            & (df_stats["strategy"] == strategy)
            & (df_stats["method"] == m)
        ].empty
    ]


def _pins_best_set(model, strategy, metric, direction):
    scored = []
    for m in _pins_methods_avail(model, strategy):
        mean, _ = _pins_lookup(model, strategy, m, metric)
        if pd.isna(mean):
            continue
        raw = abs(mean) if direction == "abs_min" else mean
        scored.append((m, round(raw, PINS_DISPLAY_DECIMALS)))
    if not scored:
        return set()
    best = min(s for _, s in scored)
    return {m for m, s in scored if s == best}


def _pins_paper_cell(mean, std, bold=False):
    # 4 LaTeX sub-cells for decimal alignment (r@{.}l +/- r@{.}l)
    if pd.isna(mean) or pd.isna(std):
        return "- & - & - & -"
    m_str = f"{round(mean, 2):.2f}"
    s_str = f"{round(std, 2):.2f}"
    m_int, m_frac = m_str.split(".")
    s_int, s_frac = s_str.split(".")
    if bold:
        return f"\\B{{{m_int}}}&\\B{{{m_frac}}} & \\B{{{s_int}}}&\\B{{{s_frac}}}"
    return f"{m_int}&{m_frac} & {s_int}&{s_frac}"


def _pins_raw_lookup(model, strategy, method, metric):
    # Raw (un-normalized) per-seed mean/std from df_all. Used for the raw-values table.
    sub = df_all[
        (df_all["dataset"] == "PinsFaceRecognition")
        & (df_all["model"] == model)
        & (df_all["strategy"] == strategy)
        & (df_all["method"] == method)
    ]
    if strategy == "random_":
        sub = sub[sub["target"] == "0.001perc"]
    if sub.empty:
        return float("nan"), float("nan")
    per_seed = sub.groupby("seed")[metric].mean()
    return float(per_seed.mean()), float(per_seed.std())


def _pins_raw_fmt(mean, std):
    # 4 LaTeX sub-cells for decimal alignment (paper-style, unbolded).
    if pd.isna(mean) or pd.isna(std):
        return "- & - & - & -"
    m_str = f"{round(mean, 2):.2f}"
    s_str = f"{round(std, 2):.2f}"
    m_int, m_frac = m_str.split(".")
    s_int, s_frac = s_str.split(".")
    return f"{m_int}&{m_frac} & {s_int}&{s_frac}"

In [ ]:
# === Build the three paper tables ===


def _build_pins_paper_table(metrics, label, caption, header_latex):
    # Generic 15-column paper-style table for 3 metrics.
    assert len(metrics) == 3
    L = []
    L.append(r"\begin{table}[t]")
    L.append(r"\caption{" + caption + r"}")
    L.append(r"\label{" + label + r"}")
    L.append(r"\centering")
    L.append(
        r"\begin{tabular}{@{}lll r@{.}l @{\,$\pm$\,} r@{.}l  r@{.}l @{\,$\pm$\,} r@{.}l  r@{.}l @{\,$\pm$\,} r@{.}l@{}}"
    )
    L.append(r"\toprule")
    header = (
        r"\textbf{Model} & \textbf{Scenario} & \textbf{Method} "
        + "".join(
            f"& \\multicolumn{{4}}{{c}}{{\\textbf{{{h}}}}} " for h in header_latex
        )
        + r"\\"
    )
    L.append(header)
    L.append(r"\midrule")

    scenarios = [("fullclass", "Full-class"), ("random_", "Random")]
    for mi, model in enumerate(["ResNet18", "ViT"]):
        per_scenario = [(s, lbl, _pins_methods_avail(model, s)) for s, lbl in scenarios]
        total = sum(len(ms) for _, _, ms in per_scenario)
        first_in_model = True
        for si, (strategy, scen_label, methods) in enumerate(per_scenario):
            n = len(methods)
            bests = {
                metric: _pins_best_set(model, strategy, metric, d)
                for metric, d in metrics
            }
            for i, method in enumerate(methods):
                if first_in_model and i == 0:
                    arch = f"\\multirow{{{total}}}{{*}}{{{model}}}"
                    first_in_model = False
                else:
                    arch = ""
                scen = (
                    f"\\multirow{{{n}}}{{*}}{{\\small {scen_label}}}" if i == 0 else ""
                )
                meth = PINS_METHOD_DISPLAY[method]
                cells = []
                for metric, _ in metrics:
                    mean, std = _pins_lookup(model, strategy, method, metric)
                    cells.append(
                        _pins_paper_cell(mean, std, bold=(method in bests[metric]))
                    )
                L.append(f"  {arch} & {scen} & {meth} & " + " & ".join(cells) + r" \\")
            if si == 0:
                L.append(r"\cmidrule{2-15}")
        if mi == 0:
            L.append(r"\midrule")

    L.append(r"\bottomrule")
    L.append(r"\end{tabular}")
    L.append(r"\end{table}")
    return "\n".join(L)


def build_pins_main_table_tex():
    headers = [
        r"$\DAcc_{\Dsetftest}$",
        r"$\DAcc_{\Dsetrtest}$",
        r"$\Layer$",
    ]
    caption = (
        r"$\DAcc$ and $\Layer$ between $\Mu$ and $\Mr$ "
        r"on Pins Face Recognition (closer to 0 is better for $\DAcc$; "
        r"lower is better for $\Layer$). "
        r"$\Layer$ is a weight-space metric with no dataset split. "
        r"Full-class values are averaged over 5 forget classes per seed; "
        r"random values use a 0.1\,\% forget set. Mean $\pm$ std across 10 seeds. "
        r"Raw values in Table~\ref{tab:pins_raw_values}. "
        r"UNSIR is excluded from the random scenario by design~\cite{tarun2023fast}."
    )
    return _build_pins_paper_table(
        MAIN_METRICS, "tab:pins_results_main", caption, headers
    )


def build_pins_appendix_table_tex():
    headers = [
        r"$\Activ_{\Dsetftest}$",
        r"$\Activ_{\Dsetrtest}$",
        r"$\DMIA$",
    ]
    caption = (
        r"$\Activ$ and $\DMIA_{\Dsetftrain}$ for full-class "
        r"and random unlearning on Pins Face Recognition. "
        r"$\DMIA_{\Dsetftrain}$ is the difference between $\Mu$ and $\Mr$ "
        r"(closer to 0 is better); $\Activ$ are raw activation distances "
        r"between $\Mu$ and $\Mr$ (lower is better). Full-class values are averaged "
        r"over 5 forget classes per seed; random values use the 0.1\,\% forget percentage. "
        r"Mean $\pm$ std across 10 seeds. Raw $\MIA$ values in "
        r"Table~\ref{tab:pins_raw_values}. "
        r"UNSIR is excluded from the random scenario by design~\cite{tarun2023fast}."
    )
    return _build_pins_paper_table(
        APPENDIX_METRICS, "tab:pins_results_appendix", caption, headers
    )


def build_pins_raw_results_tex():
    # Raw M_u and M_r Acc/MIA values per (model, scenario, method).
    L = []
    L.append(r"\begin{table}[t]")
    L.append(
        r"\caption{Raw $\Acc$ and $\MIA$ values for $\Mu$ and $\Mr$ "
        r"underlying the differences in Table~\ref{tab:pins_results_main} and "
        r"Table~\ref{tab:pins_results_appendix} for full-class and random unlearning on "
        r"Pins Face Recognition. Accuracies are reported on the test forget split "
        r"$D_f'$ and the test retain split $D_r'$; $\MIA$ is computed on the "
        r"training forget split $\Dsetftrain$. Full-class values are first averaged over "
        r"the 5 forget classes per seed; random values are reported for the "
        r"0.1\% forget percentage only. All values are then reported as mean $\pm$ std "
        r"across 10 seeds (260--269). UNSIR is excluded from the random scenario by "
        r"design~\cite{tarun2023fast}.}"
    )
    L.append(r"\label{tab:pins_raw_values}")
    L.append(r"\centering")
    L.append(r"\resizebox{\textwidth}{!}{%")
    L.append(
        r"\begin{tabular}{@{}lll r@{.}l @{\,$\pm$\,} r@{.}l  r@{.}l @{\,$\pm$\,} r@{.}l  r@{.}l @{\,$\pm$\,} r@{.}l@{}}"
    )
    L.append(r"\toprule")
    L.append(
        r"\textbf{Model} & \textbf{Scenario} & \textbf{Method} "
        r"& \multicolumn{4}{c}{\textbf{$\Acc_{\Dsetftest}$}} "
        r"& \multicolumn{4}{c}{\textbf{$\Acc_{\Dsetrtest}$}} "
        r"& \multicolumn{4}{c}{\textbf{$\MIA$}} \\"
    )
    L.append(r"\midrule")

    scenarios = [("fullclass", "Full-class"), ("random_", "Random")]
    for mi, model in enumerate(["ResNet18", "ViT"]):
        per_scenario = [(s, lbl, _pins_methods_avail(model, s)) for s, lbl in scenarios]
        total = sum(1 + len(ms) for _, _, ms in per_scenario)
        first_in_model = True
        for si, (strategy, scen_label, methods) in enumerate(per_scenario):
            block_rows = 1 + len(methods)
            if first_in_model:
                arch = f"\\multirow{{{total}}}{{*}}{{{model}}}"
                first_in_model = False
            else:
                arch = ""
            scen = f"\\multirow{{{block_rows}}}{{*}}{{\\small {scen_label}}}"
            ref_vals = [
                _pins_raw_fmt(*_pins_raw_lookup(model, strategy, "retrain", m))
                for m in RAW_METRICS
            ]
            L.append(
                f"  {arch} & {scen} & --- reference ($\\Mr$) & "
                + " & ".join(ref_vals)
                + r" \\"
            )
            for method in methods:
                meth = PINS_METHOD_DISPLAY[method]
                vals = [
                    _pins_raw_fmt(*_pins_raw_lookup(model, strategy, method, m))
                    for m in RAW_METRICS
                ]
                L.append(f"   &  & {meth} & " + " & ".join(vals) + r" \\")
            if si == 0:
                L.append(r"\cmidrule{2-15}")
        if mi == 0:
            L.append(r"\midrule")

    L.append(r"\bottomrule")
    L.append(r"\end{tabular}%")
    L.append(r"}")
    L.append(r"\end{table}")
    return "\n".join(L)

In [ ]:
# === Write the three .tex files ===

for name, builder in [
    ("pins_results_table.tex", build_pins_main_table_tex),
    ("pins_appendix_table.tex", build_pins_appendix_table_tex),
    ("pins_raw_results_table.tex", build_pins_raw_results_tex),
]:
    tex = builder()
    Path(name).write_text(tex)
    print(f"  wrote {name}  ({len(tex)} chars)")